🚀 Project Topic: Predicting Insecure Home Routers Based on Public CVE Data & Network Configuration Risks

✅ No need to create accounts on third-party services

✅ Uses open, publicly available datasets (ethical & legal)

✅ Still addresses real-world cybersecurity risks

✅ Applies Machine Learning to classify insecure routers

📌 Project Goal

We are going to ANALYZE public router vulnerabilites and predict how insecure a given router model is based on:

- Known CVEs (Common Vulnerabilities and Exposures)

- Router firmware age & update history

- Common misconfigurations (e.g., open ports, weak encryption)

- Manufacturer security  reputation (some vendors patch security vulnerabilites faster than others)

🚀 Step 1: Collect Router Security Vulnerability Data from NVD (NIST) API
Since we are focusing on router security, we will start by fetching real-world router vulnerabilities using the NVD API (National Vulnerability Database, by NIST).

The NVD API provides public Common Vulnerabilities and Exposures (CVEs).
We can search vulnerabilities by keyword, vendor, or product.

requests library -> Fetching data from the NVD API

json (built-in library) -> Store vulnerability data.

pandas -> Formats data for analysis

In [5]:
# Now, create a Python script to pull CVEs for popular router brands.

import requests
import json
import os
import pandas as pd

In [6]:
import requests
import json
import os
import time
import pandas as pd

# ✅ NVD API URL
CVE_API_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"

# ✅ List of router vendors
router_vendors = [
    "cisco", "tp-link", "netgear", "d-link", "asus", 
    "huawei", "belkin", "ubiquiti", "zyxel", "linksys"
]

# ✅ Dictionary to store CVE data
router_cve_data = {}

print("🔍 Fetching CVE data for routers...")

# ✅ Public API Rate Limit: 5 requests per 30 seconds
REQUEST_LIMIT = 5  # Public limit
TIME_WINDOW = 30  # Rolling window (30s)
request_count = 0
start_time = time.time()

# ✅ Function to fetch CVE data with rate-limit handling
def fetch_cve_data(vendor, max_retries=5):
    global request_count, start_time

    params = {
        "keywordSearch": vendor,
        "resultsPerPage": 200,  # ✅ Get more CVEs per request
    }

    retry_count = 0
    wait_time = 5  # Start with 5 seconds wait

    while retry_count < max_retries:
        # ✅ Check if we need to pause to avoid hitting the rate limit
        if request_count >= REQUEST_LIMIT:
            elapsed_time = time.time() - start_time
            if elapsed_time < TIME_WINDOW:
                sleep_time = TIME_WINDOW - elapsed_time
                print(f"⏳ Rate limit reached. Sleeping for {sleep_time:.2f} seconds...")
                time.sleep(sleep_time)

            # ✅ Reset counter after sleeping
            request_count = 0
            start_time = time.time()

        try:
            response = requests.get(CVE_API_URL, params=params)

            if response.status_code == 200:
                request_count += 1  # ✅ Track request count
                return response.json().get("vulnerabilities", [])

            elif response.status_code == 403:
                print(f"⚠️ Rate limit reached for {vendor}. Waiting {wait_time}s before retrying...")
                time.sleep(wait_time)
                retry_count += 1
                wait_time *= 2  # Exponential backoff (5s → 10s → 20s)

            else:
                print(f"❌ Failed to fetch data for {vendor}. HTTP {response.status_code}")
                return []

        except requests.RequestException as e:
            print(f"❌ Request error for {vendor}: {e}")
            return []

    print(f"⏭️ Skipping {vendor} after {max_retries} failed attempts.")
    return []  # Return empty list after multiple failures

# ✅ Fetch CVEs for each router vendor
for vendor in router_vendors:
    print(f"📡 Searching CVEs for {vendor}...")
    router_cve_data[vendor] = fetch_cve_data(vendor)

# ✅ Save CVE data as JSON
os.makedirs("data", exist_ok=True)
with open("data/router_cve_data.json", "w") as file:
    json.dump(router_cve_data, file, indent=4)

print("✅ Router vulnerability data saved in 'data/router_cve_data.json'.")

# ✅ Convert data to a DataFrame
cve_list = []
for vendor, cves in router_cve_data.items():
    for cve in cves:
        cve_info = cve.get("cve", {})
        cve_list.append({
            "Vendor": vendor,
            "CVE_ID": cve_info.get("id", "N/A"),
            "Description": cve_info.get("descriptions", [{}])[0].get("value", "N/A"),
            "Severity": cve_info.get("metrics", {}).get("cvssMetricV2", [{}])[0].get("baseSeverity", "N/A"),
            "Score": cve_info.get("metrics", {}).get("cvssMetricV2", [{}])[0].get("cvssData", {}).get("baseScore", "N/A"),
            "Published": cve_info.get("published", "N/A"),
            "Last Modified": cve_info.get("lastModified", "N/A"),
        })

df = pd.DataFrame(cve_list)

# ✅ Save to CSV
df.to_csv("data/router_cve_data.csv", index=False)

print("✅ Router CVE data saved in 'data/router_cve_data.csv'.")



🔍 Fetching CVE data for routers...
📡 Searching CVEs for cisco...
📡 Searching CVEs for tp-link...
📡 Searching CVEs for netgear...
📡 Searching CVEs for d-link...
📡 Searching CVEs for asus...
📡 Searching CVEs for huawei...
📡 Searching CVEs for belkin...
📡 Searching CVEs for ubiquiti...
📡 Searching CVEs for zyxel...
📡 Searching CVEs for linksys...
✅ Router vulnerability data saved in 'data/router_cve_data.json'.
✅ Router CVE data saved in 'data/router_cve_data.csv'.


In [7]:
print(df["Severity"])

0       MEDIUM
1       MEDIUM
2       MEDIUM
3       MEDIUM
4         HIGH
         ...  
1657       N/A
1658       N/A
1659       N/A
1660       N/A
1661       N/A
Name: Severity, Length: 1662, dtype: object


In [8]:
import requests
import time

# ✅ NVD API URL
CVE_API_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"

# ✅ Function to fetch missing severity
def fetch_severity(cve_id):
    params = {"cveId": cve_id}
    response = requests.get(CVE_API_URL, params=params)

    if response.status_code == 200:
        data = response.json()
        vulnerabilities = data.get("vulnerabilities", [])
        
        if vulnerabilities:
            vuln = vulnerabilities[0]["cve"]
            if "metrics" in vuln:
                metrics = vuln["metrics"]

                # ✅ Try CVSS v3.1
                if "cvssMetricV31" in metrics and metrics["cvssMetricV31"]:
                    return metrics["cvssMetricV31"][0]["cvssData"]["baseSeverity"]

                # ✅ Try CVSS v3
                elif "cvssMetricV3" in metrics and metrics["cvssMetricV3"]:
                    return metrics["cvssMetricV3"][0]["cvssData"]["baseSeverity"]

                # ✅ Try CVSS v2
                elif "cvssMetricV2" in metrics and metrics["cvssMetricV2"]:
                    return metrics["cvssMetricV2"][0]["baseSeverity"]

    return "Unknown"  # If severity is still not found

# ✅ Process missing severity rows
missing_cves = df[df["Severity"].isna()]["CVE_ID"].tolist()

for idx, cve_id in enumerate(missing_cves):
    new_severity = fetch_severity(cve_id)
    df.loc[df["CVE_ID"] == cve_id, "Severity"] = new_severity

    # ✅ Print progress
    if idx % 10 == 0:
        print(f"🔄 Updated {idx}/{len(missing_cves)} missing severity values...")

    time.sleep(1)  # ✅ Prevent rate limits

# ✅ Save updated dataset
df.to_csv("data/router_cve_data_updated.csv", index=False)

print("\n✅ Missing severity values have been updated.")


print(df["Severity"])


✅ Missing severity values have been updated.
0       MEDIUM
1       MEDIUM
2       MEDIUM
3       MEDIUM
4         HIGH
         ...  
1657       N/A
1658       N/A
1659       N/A
1660       N/A
1661       N/A
Name: Severity, Length: 1662, dtype: object


In [9]:
print(df["Severity"])

0       MEDIUM
1       MEDIUM
2       MEDIUM
3       MEDIUM
4         HIGH
         ...  
1657       N/A
1658       N/A
1659       N/A
1660       N/A
1661       N/A
Name: Severity, Length: 1662, dtype: object


In [10]:
print("\n🔍 Missing Values Summary:")
print(df[["Severity", "Score"]].isna().sum())



🔍 Missing Values Summary:
Severity    0
Score       0
dtype: int64


In [11]:
print("\n🔍 Missing Values Per Vendor:")
print(df[df["Severity"].isna()]["Vendor"].value_counts())



🔍 Missing Values Per Vendor:
Series([], Name: count, dtype: int64)


In [12]:
import requests
import time

CVE_API_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"

def fetch_severity(cve_id):
    params = {"cveId": cve_id}
    response = requests.get(CVE_API_URL, params=params)

    if response.status_code == 200:
        data = response.json()
        vulnerabilities = data.get("vulnerabilities", [])

        if vulnerabilities:
            vuln = vulnerabilities[0]["cve"]
            if "metrics" in vuln:
                metrics = vuln["metrics"]

                # ✅ Try CVSS v3.1
                if "cvssMetricV31" in metrics and metrics["cvssMetricV31"]:
                    return metrics["cvssMetricV31"][0]["cvssData"]["baseSeverity"]

                # ✅ Try CVSS v3
                elif "cvssMetricV3" in metrics and metrics["cvssMetricV3"]:
                    return metrics["cvssMetricV3"][0]["cvssData"]["baseSeverity"]

                # ✅ Try CVSS v2
                elif "cvssMetricV2" in metrics and metrics["cvssMetricV2"]:
                    return metrics["cvssMetricV2"][0]["baseSeverity"]

    return "Unknown"  # If severity is still missing

# ✅ Process missing severity rows for affected vendors
affected_vendors = ["zyxel", "linksys", "ubiquiti", "belkin"]
missing_cves = df[df["Vendor"].isin(affected_vendors) & df["Severity"].isna()]["CVE_ID"].tolist()

for idx, cve_id in enumerate(missing_cves):
    new_severity = fetch_severity(cve_id)
    df.loc[df["CVE_ID"] == cve_id, "Severity"] = new_severity

    if idx % 10 == 0:
        print(f"🔄 Updated {idx}/{len(missing_cves)} missing severity values...")

    time.sleep(1)  # Prevent rate limits

# ✅ Save updated dataset
df.to_csv("data/router_cve_data_updated.csv", index=False)
print("\n✅ Missing severity values have been updated.")



✅ Missing severity values have been updated.


In [13]:
print("\n🔍 Missing Values Per Vendor:")
print(df[df["Severity"].isna()]["Vendor"].value_counts())


🔍 Missing Values Per Vendor:
Series([], Name: count, dtype: int64)


In [14]:
# Filter the dataset for rows where Severity is missing
missing_severity_df = df[df["Severity"].isna()]

# Print the first few missing rows
print("\n🔍 Preview of Missing Severity CVEs:")
print(missing_severity_df[["Vendor", "CVE_ID", "Severity"]].head(10))

# Print all missing CVEs grouped by vendor
for vendor in ["zyxel", "linksys", "ubiquiti", "belkin"]:
    print(f"\n🔍 Missing Severity for {vendor.upper()}:")
    print(missing_severity_df[missing_severity_df["Vendor"] == vendor][["CVE_ID", "Severity"]])



🔍 Preview of Missing Severity CVEs:
Empty DataFrame
Columns: [Vendor, CVE_ID, Severity]
Index: []

🔍 Missing Severity for ZYXEL:
Empty DataFrame
Columns: [CVE_ID, Severity]
Index: []

🔍 Missing Severity for LINKSYS:
Empty DataFrame
Columns: [CVE_ID, Severity]
Index: []

🔍 Missing Severity for UBIQUITI:
Empty DataFrame
Columns: [CVE_ID, Severity]
Index: []

🔍 Missing Severity for BELKIN:
Empty DataFrame
Columns: [CVE_ID, Severity]
Index: []


In [15]:
import requests
import time

CVE_API_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"

# ✅ Function to fetch severity for a specific CVE ID
def fetch_severity(cve_id):
    params = {"cveId": cve_id}
    response = requests.get(CVE_API_URL, params=params)

    if response.status_code == 200:
        data = response.json()
        vulnerabilities = data.get("vulnerabilities", [])

        if vulnerabilities:
            vuln = vulnerabilities[0]["cve"]
            if "metrics" in vuln:
                metrics = vuln["metrics"]

                # ✅ Try CVSS v3.1
                if "cvssMetricV31" in metrics and metrics["cvssMetricV31"]:
                    return metrics["cvssMetricV31"][0]["cvssData"]["baseSeverity"]

                # ✅ Try CVSS v3
                elif "cvssMetricV3" in metrics and metrics["cvssMetricV3"]:
                    return metrics["cvssMetricV3"][0]["cvssData"]["baseSeverity"]

                # ✅ Try CVSS v2
                elif "cvssMetricV2" in metrics and metrics["cvssMetricV2"]:
                    return metrics["cvssMetricV2"][0]["baseSeverity"]

    return "Unknown"  # If severity is still missing

# ✅ Get a list of missing CVEs
missing_cves = df[df["Severity"].isna()]["CVE_ID"].tolist()

# ✅ Fetch severity for missing CVEs
for idx, cve_id in enumerate(missing_cves):
    new_severity = fetch_severity(cve_id)
    df.loc[df["CVE_ID"] == cve_id, "Severity"] = new_severity

    # ✅ Print progress every 10 CVEs
    if idx % 10 == 0:
        print(f"🔄 Updated {idx}/{len(missing_cves)} missing severity values...")

    time.sleep(1)  # ✅ Prevent rate limits

# ✅ Save updated dataset
df.to_csv("data/router_cve_data_updated.csv", index=False)
print("\n✅ Missing severity values have been updated.")



✅ Missing severity values have been updated.


In [16]:
import pandas as pd

# Load the dataset
df = pd.read_csv("data/router_cve_data.csv")

# Display all unique values in the Severity column
print("\n🔍 Unique Values in Severity Column:")
print(df["Severity"].unique())

# Print all values in the Severity column (every row)
print("\n🔍 Full List of Severity Values:")
print(df["Severity"].to_list())  # Converts the column to a list for easy viewing



🔍 Unique Values in Severity Column:
['MEDIUM' 'HIGH' 'LOW' nan]

🔍 Full List of Severity Values:
['MEDIUM', 'MEDIUM', 'MEDIUM', 'MEDIUM', 'HIGH', 'HIGH', 'MEDIUM', 'MEDIUM', 'MEDIUM', 'HIGH', 'HIGH', 'MEDIUM', 'MEDIUM', 'MEDIUM', 'MEDIUM', 'HIGH', 'HIGH', 'MEDIUM', 'HIGH', 'MEDIUM', 'MEDIUM', 'LOW', 'LOW', 'HIGH', 'LOW', 'HIGH', 'HIGH', 'HIGH', 'HIGH', 'HIGH', 'HIGH', 'HIGH', 'HIGH', 'MEDIUM', 'MEDIUM', 'LOW', 'LOW', 'HIGH', 'MEDIUM', 'MEDIUM', 'MEDIUM', 'HIGH', 'MEDIUM', 'HIGH', 'MEDIUM', 'HIGH', 'HIGH', 'HIGH', 'LOW', 'LOW', 'HIGH', 'MEDIUM', 'HIGH', 'MEDIUM', 'MEDIUM', 'MEDIUM', 'MEDIUM', 'MEDIUM', 'HIGH', 'MEDIUM', 'HIGH', 'HIGH', 'MEDIUM', 'MEDIUM', 'LOW', 'HIGH', 'HIGH', 'MEDIUM', 'HIGH', 'HIGH', 'MEDIUM', 'HIGH', 'MEDIUM', 'LOW', 'MEDIUM', 'MEDIUM', 'HIGH', 'MEDIUM', 'HIGH', 'MEDIUM', 'HIGH', 'MEDIUM', 'MEDIUM', 'HIGH', 'MEDIUM', 'HIGH', 'HIGH', 'HIGH', 'HIGH', 'MEDIUM', 'HIGH', 'MEDIUM', 'MEDIUM', 'MEDIUM', 'MEDIUM', 'MEDIUM', 'MEDIUM', 'MEDIUM', 'LOW', 'HIGH', 'MEDIUM', 'MEDI

In [17]:
# Count non-empty (non-null) values in the Severity column
non_empty_severity_count = df["Severity"].notnull().sum()

print(f"\n✅ Total Non-Empty Severity Values: {non_empty_severity_count}")



✅ Total Non-Empty Severity Values: 1554


In [18]:
# Find rows where Severity is empty or NaN
missing_severity_rows = df[df["Severity"].isnull() | (df["Severity"].str.strip() == "")]

print("\n🔍 Rows with Missing Severity Values:")
print(missing_severity_rows)


🔍 Rows with Missing Severity Values:
        Vendor          CVE_ID  \
1258    belkin  CVE-2023-27217   
1259    belkin  CVE-2023-33768   
1260    belkin  CVE-2024-53223   
1287  ubiquiti  CVE-2023-23119   
1288  ubiquiti  CVE-2023-24104   
...        ...             ...   
1657   linksys  CVE-2024-57542   
1658   linksys  CVE-2024-57543   
1659   linksys  CVE-2024-57544   
1660   linksys  CVE-2024-57545   
1661   linksys  CVE-2025-25522   

                                            Description Severity  Score  \
1258  A stack-based buffer overflow in the ChangeFri...      NaN    NaN   
1259  Incorrect signature verification of the firmwa...      NaN    NaN   
1260  In the Linux kernel, the following vulnerabili...      NaN    NaN   
1287  The use of the cyclic redundancy check (CRC) a...      NaN    NaN   
1288  Ubiquiti Networks UniFi Dream Machine Pro v7.2...      NaN    NaN   
...                                                 ...      ...    ...   
1657  Linksys E8450 v1.2.00.

In [19]:
print("\n🔍 Missing Values Summary (After Re-Fetching):")
print(df[["Severity", "Score"]].isna().sum())



🔍 Missing Values Summary (After Re-Fetching):
Severity    108
Score       108
dtype: int64


In [20]:
def assign_severity(row):
    if pd.isna(row["Severity"]) and not pd.isna(row["Score"]):  
        score = row["Score"]
        if score <= 3.9:
            return "LOW"
        elif score <= 6.9:
            return "MEDIUM"
        elif score <= 8.9:
            return "HIGH"
        else:
            return "CRITICAL"
    return row["Severity"]  # Keep existing Severity if available

df["Severity"] = df.apply(assign_severity, axis=1)


In [21]:
severity_score_mapping = {
    "LOW": 3.0,
    "MEDIUM": 5.5,
    "HIGH": 8.0,
    "CRITICAL": 9.5
}

df["Score"] = df.apply(lambda row: severity_score_mapping[row["Severity"]] if pd.isna(row["Score"]) and row["Severity"] in severity_score_mapping else row["Score"], axis=1)


In [22]:
df.loc[(df["Severity"].isna()) & (df["Published"].str.contains("2024|2025")), "Severity"] = "Pending"


In [23]:
df = df.copy()  # ✅ Ensure we modify the original DataFrame
df["Severity"] = df["Severity"].fillna("Unknown")
df["Score"] = df["Score"].fillna(0.0)



In [24]:
print("\n✅ Final Missing Values Check:")
print(df[["Severity", "Score"]].isna().sum())



✅ Final Missing Values Check:
Severity    0
Score       0
dtype: int64


In [25]:
df.to_csv("data/router_cve_data_final.csv", index=False)
print("\n✅ Final cleaned dataset saved as 'data/router_cve_data_final.csv'.")



✅ Final cleaned dataset saved as 'data/router_cve_data_final.csv'.


In [26]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get the API key
api_key = os.getenv("NVD_API_Key")






In [3]:
import requests
import os
import pandas as pd
import time
from dotenv import load_dotenv

# Load API key from .env file
load_dotenv()
NVD_API_KEY = os.getenv("NVD_API_KEY")

# Load dataset
df = pd.read_csv("data/router_cve_data_final.csv")

# Filter out very old CVEs (pre-2000) since they often return errors
df = df[df["CVE_ID"].str.startswith("CVE-20")]  # Keeps CVE-2000 and later only

# Extract unique CVE IDs
cve_ids = df["CVE_ID"].dropna().unique().tolist()

# API Rate Limits
REQUEST_INTERVAL = 2  # Seconds between each request
TIME_LIMIT = 60  # Max runtime for requests
start_time = time.time()

# Log failed requests separately
failed_requests = []

# Function to fetch CVE data with improved retries
def fetch_cve_data(cve_id, max_retries=5):
    url = f"https://services.nvd.nist.gov/rest/json/cves/2.0?cveId={cve_id}"
    headers = {"apiKey": NVD_API_KEY}
    
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url, headers=headers, timeout=5)  # Reduced timeout to 5s
            
            if response.status_code == 200:
                data = response.json()
                product, version = None, None

                # Extract product/version information
                try:
                    product_info = data["vulnerabilities"][0]["cve"]["configurations"][0]["nodes"][0]["cpeMatch"][0]
                    product = product_info.get("criteria", "Unknown")
                    version = product_info.get("versionEndIncluding", "Unknown")
                except (KeyError, IndexError):
                    print(f"⚠️ No product/version found for {cve_id}, skipping.")

                return {"CVE_ID": cve_id, "Product": product, "Version": version, "Status": "✅ Success"}

            elif response.status_code in [403, 404, 503]:
                error_message = f"❌ API Error {response.status_code}"
                failed_requests.append({"CVE_ID": cve_id, "Error": error_message})
                return {"CVE_ID": cve_id, "Product": None, "Version": None, "Status": error_message}

            else:
                return {"CVE_ID": cve_id, "Product": None, "Version": None, "Status": f"❌ API Error {response.status_code}"}

        except requests.RequestException as e:
            print(f"⚠️ Network error for {cve_id}: {e}, retrying in {2 ** retries} sec...")
            time.sleep(2 ** retries)  # Exponential backoff (1s, 2s, 4s, ...)
            retries += 1

    # If max retries reached, log failure
    failed_requests.append({"CVE_ID": cve_id, "Error": "⚠️ Max retries reached"})
    return {"CVE_ID": cve_id, "Product": None, "Version": None, "Status": "⚠️ Max retries reached"}

# Process CVEs while respecting the time limit
results = []
for cve_id in cve_ids[:30]:  # Limit to 30 CVEs for testing
    elapsed_time = time.time() - start_time
    if elapsed_time > TIME_LIMIT:
        print("⏳ Time limit reached. Stopping requests.")
        break
    
    print(f"🔍 Fetching data for {cve_id}...")
    result = fetch_cve_data(cve_id)
    results.append(result)
    
    time.sleep(REQUEST_INTERVAL)  # Wait before next request

# Convert results to DataFrame and merge with original dataset
df_results = pd.DataFrame(results)
df = df.merge(df_results, on="CVE_ID", how="left")

# Save updated dataset
df.to_csv("data/router_cve_data_final_with_product.csv", index=False)
print("\n✅ Updated dataset saved as 'data/router_cve_data_final_with_product.csv'.")

# Save failed requests separately
if failed_requests:
    df_failed = pd.DataFrame(failed_requests)
    df_failed.to_csv("data/nvd_failed_requests.csv", index=False)
    print("\n⚠️ Some requests failed. See 'data/nvd_failed_requests.csv' for details.")


🔍 Fetching data for CVE-2000-0267...
🔍 Fetching data for CVE-2000-0268...
⚠️ Network error for CVE-2000-0268: HTTPSConnectionPool(host='services.nvd.nist.gov', port=443): Read timed out. (read timeout=5), retrying in 1 sec...
🔍 Fetching data for CVE-2000-0345...
🔍 Fetching data for CVE-2000-0368...
🔍 Fetching data for CVE-2000-0380...
🔍 Fetching data for CVE-2000-0486...
🔍 Fetching data for CVE-2000-0613...
⚠️ Network error for CVE-2000-0613: HTTPSConnectionPool(host='services.nvd.nist.gov', port=443): Read timed out. (read timeout=5), retrying in 1 sec...
⚠️ Network error for CVE-2000-0613: HTTPSConnectionPool(host='services.nvd.nist.gov', port=443): Read timed out. (read timeout=5), retrying in 2 sec...
⚠️ Network error for CVE-2000-0613: HTTPSConnectionPool(host='services.nvd.nist.gov', port=443): Read timed out. (read timeout=5), retrying in 4 sec...
⚠️ Network error for CVE-2000-0613: HTTPSConnectionPool(host='services.nvd.nist.gov', port=443): Read timed out. (read timeout=5), re